In [ ]:
import sys
import os

# Add the src directory to the system path to allow importing custom modules
sys.path.insert(0, os.path.abspath('../src'))

import warnings
warnings.filterwarnings('ignore')

# Enable autoreload to automatically reload modules when they are edited
%load_ext autoreload
%autoreload 2

import numpy as np
import pickle
import pandas as pd

from fetch_prepare_data import *
from train_model_predict import *

# Set a consistent style for all plots
import matplotlib.pyplot as plt
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})


# complete machine learning pipeline

* read and prepare data from downloaded csv from Kaggle
* fetch and prepare weather data from open-meteo API
* combine and create features for energy and weather data
* train and test data split - time series split
* train and evaluate model
* GridSearchCV and BayesSearchCV
* save best model 

* dataframe preparated by function differs from the old one from csv, use df_old.compare(df_new) to check differences

In [ ]:
from fetch_prepare_data import *
from train_model_predict import *

df_for_modeling = prepare_data_for_modeling()
print(df_for_modeling.shape)
display(df_for_modeling.head(3))    

df_for_modeling.to_pickle('../data/processed/energy_weather_data_for_modeling.pkl')

In [ ]:
# tune LightGBM with Bayesian optimization using Optuna
# tune without preprocessor - tree models are not affected by scaling, so we can skip the preprocessor for tuning
from lightgbm import LGBMRegressor
import pandas as pd
from fetch_prepare_data import *
from train_model_predict import *

#with open('../data/processed/energy_weather_data_for_modeling.pkl', 'rb') as f:
#    df_for_modeling = pickle.load(f)
#df_for_modeling = prepare_data_for_modeling()

features_train, target_train, features_test, target_test = train_test_split_by_date(df_for_modeling, 
                                                                                    date_column='time',
                                                                                    target_column='EnergyDemand', 
                                                                                    split_date='2025-01-01')

# define continuous parameter grid for LightGBM
param_lgbm_continuous = { 
    'n_estimators': (50, 500),  # range for n_estimators
    'learning_rate': (0.01, 0.3),  # range for learning_rate
    'max_depth': (3, 15)  # range for max_depth
}

model_lgbm =LGBMRegressor(random_state=42, force_col_wise=True)
best_model_lgbm, best_params_lgbm = tune_model_bayesian(
    model_pipeline=model_lgbm, 
    in_param_bayes=param_lgbm_continuous,
    in_features_train=features_train, 
    in_target_train=target_train)
print(f"Best hyperparameters: {best_params_lgbm}")

print()

prediction_lgbm = best_model_lgbm.predict(features_test)

print_scores('LightGBM', target_test, prediction_lgbm)  

save_model_to_pickle(best_model_lgbm, '../models/best_lgbm_model_bayesian.pkl')


In [ ]:
# tune XGBoost hyperparameters using Bayesian optimization with Optuna
from xgboost import XGBRegressor
from fetch_prepare_data import *
from train_model_predict import *

#with open('../data/processed/energy_weather_data_for_modeling.pkl', 'rb') as f:
#    df_for_modeling = pickle.load(f)
#features_train, target_train, features_test, target_test = train_test_split_by_date(df_for_modeling, 
#                                                                                    date_column='time',
#                                                                                    target_column='EnergyDemand', 
#                                                                                    split_date='2025-01-01')

# define continuous parameter grid for XGBoost
param_xgb_continuous = {
    'n_estimators': (50, 500),  # range for n_estimators
    'max_depth': (3, 15),       # range for max_depth
    'learning_rate': (0.01, 0.3), # range for learning_rate
    'subsample': (0.5, 1.0),    # range for subsample
    'colsample_bytree': (0.5, 1.0), # range for colsample_bytree
    # max_features is categorical
    'max_features': [1.0, 'sqrt', 'log2'],
}

model_xgb = XGBRegressor(random_state=42)
best_model_xgb, best_params_xgb = tune_model_bayesian(
    model_pipeline=model_xgb,
    in_param_bayes=param_xgb_continuous,
    in_features_train=features_train,   
    in_target_train=target_train
)
print(f"Best hyperparameters for XGBoost: {best_params_xgb}")

y_pred_xgb = best_model_xgb.predict(features_test)

print_scores('XGBoost', target_test, y_pred_xgb)

save_model_to_pickle(best_model_xgb, '../models/best_xgb_model_bayesian.pkl')

In [ ]:
# Skipped 
# - give up Random Forest tuning 
# - it takes too long and the performance is not improving significantly compared to LightGBM and XGBoost

# tune Random Forest with Bayesian optimization using Optuna
#features_train, target_train, features_test, target_test = train_test_split_by_date(df_for_modeling, 
#                                                                                    date_column='time',
#                                                                                    target_column='EnergyDemand', 
#                                                                                    split_date='2025-01-01')

# define continuous parameter grid for Random Forest
param_rf_continuous = {
    'n_estimators': (50, 500),  # range for n_estimators
    'max_depth': (3, 15),       # range for max_depth
    'min_samples_split': (2, 10), # range for min_samples_split
    'min_samples_leaf': (1, 4),   # range for min_samples_leaf
    # max_features is categorical
    'max_features': [1.0, 'sqrt', 'log2'],
}

model_rf = RandomForestRegressor(random_state=42)
best_model_rf, best_params_rf = tune_model_bayesian(
    model_pipeline=model_rf,
    in_param_bayes=param_rf_continuous,
    in_features_train=features_train,   
    in_target_train=target_train
)
print(f"Best hyperparameters for Random Forest: {best_params_rf}")

y_pred_rf = best_model_rf.predict(features_test)

print_scores('RandomForest', target_test, y_pred_rf)

save_model_to_pickle(best_model_rf, '../models/best_rf_model_bayesian.pkl')



In [ ]:
# load the best models from pickle files and prepare the data for modeling
best_model_lgbm = load_model_from_pickle('../models/best_lgbm_model_bayesian.pkl')
best_model_xgb = load_model_from_pickle('../models/best_xgb_model_bayesian.pkl')
#best_model_rf = load_model_from_pickle('../models/best_rf_model_bayesian.pkl')

# load the data for modeling from pickle file
#with open('../data/processed/energy_weather_data_for_modeling.pkl', 'rb') as f:
#    data_for_modeling = pickle.load(f)
#features_train, target_train, features_test, target_test = train_test_split_by_date(data_for_modeling, 
#                                                                                    date_column='time',
#                                                                                    target_column='EnergyDemand', 
#                                                                                    split_date='2025-01-01')


In [ ]:
# plot learning curve for LGBM and Random Forest models
plot_learning_curve(best_model_lgbm, 'LGBM model', features_train, target_train)
plot_learning_curve(best_model_xgb, 'XGBoost model', features_train, target_train)
#plot_learning_curve(best_model_rf, 'Random Forest model', features_train, target_train)